# Breadfruit Maturity & Quality Classifier

Multi-task training with EfficientNet-B0 — predicts both maturity (immature / partially_mature / mature)
and quality (poor / good / excellent) from a single image.

---
### Setup

Before running this notebook:
1. Run `python labeling_tool/package_for_colab.py` on your local machine
2. Upload `breadfruit_classifier.zip` and `training_dataset.zip` to Google Drive
3. Update the `DRIVE_ZIP_DIR` path below to point to where you uploaded them

In [ ]:
# @title 1. Mount Google Drive & install dependencies
from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURE THIS ---
# Path to the folder containing your ZIP files on Google Drive
DRIVE_ZIP_DIR = "/content/drive/MyDrive/breadfruit"
# -----------------------

In [ ]:
# @title 2. Extract ZIP files
import zipfile
import os

os.makedirs("/content/breadfruit_classifier", exist_ok=True)
os.makedirs("/content/training_dataset", exist_ok=True)

print("Extracting breadfruit_classifier.zip...")
with zipfile.ZipFile(os.path.join(DRIVE_ZIP_DIR, "breadfruit_classifier.zip"), "r") as z:
    z.extractall("/content/")

print("Extracting training_dataset.zip...")
with zipfile.ZipFile(os.path.join(DRIVE_ZIP_DIR, "training_dataset.zip"), "r") as z:
    z.extractall("/content/")

print("\nExtracted files:")
print(f"  Classifier: {os.listdir('/content/breadfruit_classifier/')}")
print(f"  Dataset splits: {os.listdir('/content/training_dataset/')}")
print(f"  Annotations: {open('/content/breadfruit_classifier/annotations.csv').readline().strip()}")

In [ ]:
# @title 3. Install Python dependencies
!pip install -q torch torchvision numpy opencv-python matplotlib seaborn scikit-learn scipy tqdm Pillow

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# @title 4. Explore the labeled dataset
import csv
from collections import Counter
from pathlib import Path

annotation_path = Path("/content/breadfruit_classifier/annotations.csv")
data_root = Path("/content/training_dataset")

# Load annotations
maturity_counts = Counter()
quality_counts = Counter()
combined_counts = Counter()
split_counts = {"train": 0, "val": 0, "test": 0}

with open(annotation_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        fname = row["filename"]
        maturity_counts[row["maturity"]] += 1
        quality_counts[row["quality"]] += 1
        combined_counts[f"{row['maturity']}-{row['quality']}"] += 1
        for split in ["train", "val", "test"]:
            if (data_root / split / "breadfruit" / fname).exists():
                split_counts[split] += 1
                break

total_annotated = sum(maturity_counts.values())
total_images = sum(1 for _ in data_root.rglob("*") if _.suffix.lower() in (".jpg", ".jpeg", ".png"))

print(f"Total images in dataset: {total_images}")
print(f"Annotated images: {total_annotated}")
print(f"Unlabeled images: {total_images - total_annotated}")
print()

print("Maturity distribution:")
for cls, cnt in maturity_counts.most_common():
    print(f"  {cls:20s}: {cnt:3d} ({cnt/total_annotated*100:.1f}%)")
print()

print("Quality distribution:")
for cls, cnt in quality_counts.most_common():
    print(f"  {cls:20s}: {cnt:3d} ({cnt/total_annotated*100:.1f}%)")
print()

print("Combined distribution (top 10):")
for combo, cnt in combined_counts.most_common(10):
    print(f"  {combo:35s}: {cnt:3d}")
print()

print(f"Annotated images per split:")
for split in ["train", "val", "test"]:
    print(f"  {split}: {split_counts[split]}")

---
## Training

The model is **EfficientNet-B0** with two classification heads:
- Maturity head (3 classes)
- Quality head (3 classes)

**Training strategy** (two-phase):
1. **Phase 1**: Freeze backbone, train only the heads (5 epochs, LR=1e-3)
2. **Phase 2**: Unfreeze backbone, fine-tune everything (LR=1e-4)

Uses AdamW optimizer, ReduceLROnPlateau scheduler, and early stopping.

In [ ]:
# @title 5. Run training
import sys
sys.path.insert(0, "/content/breadfruit_classifier")

from config import Config
from main import cmd_train, build_parser

# Build the config and override paths for Colab
config = Config()
config.data.data_root = "/content/training_dataset"
config.data.annotation_file = "/content/breadfruit_classifier/annotations.csv"
config.logging.checkpoint_dir = "/content/breadfruit_classifier/checkpoints"
config.logging.log_dir = "/content/breadfruit_classifier/logs"
config.logging.plot_dir = "/content/breadfruit_classifier/plots"
config.training.num_workers = 2  # Colab has limited CPUs

# Parse minimal args for the cmd_train function
parser = build_parser()
args = parser.parse_args(["train"])
args.epochs = None
args.batch_size = None
args.lr = None
args.experiment = "breadfruit_colab"
args.data_root = None
args.annotation_file = None

# Run training
cmd_train(config, args)

---
## Results & Visualization

In [ ]:
# @title 6. View training curves
from IPython.display import Image as IPImage, display
from pathlib import Path

plot_dir = Path("/content/breadfruit_classifier/plots")
for png in sorted(plot_dir.glob("*.png")):
    print(f"\n{png.name}")
    display(IPImage(filename=str(png)))

In [ ]:
# @title 7. Evaluate on test set
import sys
sys.path.insert(0, "/content/breadfruit_classifier")

from config import Config
from datasets import get_dataloaders
from model import create_model
from train import load_checkpoint
from evaluate import evaluate_model, print_evaluation_report, plot_confusion_matrices

config = Config()
config.data.data_root = "/content/training_dataset"
config.data.annotation_file = "/content/breadfruit_classifier/annotations.csv"
config.logging.plot_dir = "/content/breadfruit_classifier/plots"
config.training.num_workers = 2

checkpoint_path = Path("/content/breadfruit_classifier/checkpoints/best_model.pth")
if not checkpoint_path.exists():
    print("No best_model.pth found. Train first!")
else:
    model, _ = load_checkpoint(str(checkpoint_path), config=config)
    _, _, test_loader = get_dataloaders(config)

    if test_loader and len(test_loader.dataset) > 0:
        print(f"Test samples: {len(test_loader.dataset)}")
        results = evaluate_model(model, test_loader, config)
        print_evaluation_report(results)
        plot_confusion_matrices(results, save_dir=config.logging.plot_dir, prefix="test_breadfruit")
    else:
        print("No labeled test data found.")

In [ ]:
# @title 8. Test confusion matrices
from IPython.display import Image as IPImage, display
from pathlib import Path

plot_dir = Path("/content/breadfruit_classifier/plots")
for png in sorted(plot_dir.glob("test_*.png")):
    print(f"\n{png.name}")
    display(IPImage(filename=str(png)))

In [ ]:
# @title 9. Predict on new images (upload your own)
import sys
sys.path.insert(0, "/content/breadfruit_classifier")
from pathlib import Path
from google.colab import files
from config import Config
from predict import load_model_for_inference, predict_single

config = Config()
config.data.data_root = "/content/training_dataset"

checkpoint_path = Path("/content/breadfruit_classifier/checkpoints/best_model.pth")
if not checkpoint_path.exists():
    print("No best_model.pth found. Train first!")
else:
    model = load_model_for_inference(str(checkpoint_path), config)
    print("\nUpload one or more breadfruit images to classify:")
    uploaded = files.upload()

    for fname in uploaded:
        result = predict_single(model, fname, config, return_probs=True)
        print(f"\n{'='*50}")
        print(f"  {fname}")
        print(f"{'='*50}")
        print(f"  Maturity: {result['maturity']['prediction']}")
        print(f"    Confidence: {result['maturity']['confidence']:.4f}")
        for cls, prob in result['maturity']['probabilities'].items():
            print(f"    {cls:20s}: {prob:.4f}")
        print(f"  Quality:  {result['quality']['prediction']}")
        print(f"    Confidence: {result['quality']['confidence']:.4f}")
        for cls, prob in result['quality']['probabilities'].items():
            print(f"    {cls:20s}: {prob:.4f}")

In [ ]:
# @title 10. Save results back to Google Drive
import shutil
from pathlib import Path

output_dir = Path(DRIVE_ZIP_DIR) / "colab_results"
output_dir.mkdir(parents=True, exist_ok=True)

# Copy checkpoints
ckpt_src = Path("/content/breadfruit_classifier/checkpoints")
if ckpt_src.exists():
    shutil.copytree(ckpt_src, output_dir / "checkpoints", dirs_exist_ok=True)
    print(f"Copied checkpoints to {output_dir / 'checkpoints'}")

# Copy plots
plots_src = Path("/content/breadfruit_classifier/plots")
if plots_src.exists():
    shutil.copytree(plots_src, output_dir / "plots", dirs_exist_ok=True)
    print(f"Copied plots to {output_dir / 'plots'}")

# Copy logs
logs_src = Path("/content/breadfruit_classifier/logs")
if logs_src.exists():
    shutil.copytree(logs_src, output_dir / "logs", dirs_exist_ok=True)
    print(f"Copied logs to {output_dir / 'logs'}")

print(f"\nAll results saved to {output_dir}")